In [ ]:
!pip install --upgrade mani_skill torch

In [ ]:
#Should be able to run now - but the video doesn't seem to work
!python -m mani_skill.examples.demo_random_action -e PushCube-v1 \
  --render-mode="rgb_array" --record-dir="videos"

In [ ]:
!pip install lerobot
!pip install -e ".[smolvla]"

In [ ]:
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
policy = SmolVLAPolicy.from_pretrained("lerobot/smolvla_base")

/home/dl01e26/.conda/envs/vla/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...


`torch_dtype` is deprecated! Use `dtype` instead!


Reducing the number of VLM layers to 16 ...


In [ ]:
for n, p in policy.named_parameters():   # inspecting the insane structure behind the model
    print(n, p.shape, p.requires_grad) 

model.vlm_with_expert.vlm.model.vision_model.embeddings.patch_embedding.weight torch.Size([768, 3, 16, 16]) False
model.vlm_with_expert.vlm.model.vision_model.embeddings.patch_embedding.bias torch.Size([768]) False
model.vlm_with_expert.vlm.model.vision_model.embeddings.position_embedding.weight torch.Size([1024, 768]) False
model.vlm_with_expert.vlm.model.vision_model.encoder.layers.0.self_attn.k_proj.weight torch.Size([768, 768]) False
model.vlm_with_expert.vlm.model.vision_model.encoder.layers.0.self_attn.k_proj.bias torch.Size([768]) False
model.vlm_with_expert.vlm.model.vision_model.encoder.layers.0.self_attn.v_proj.weight torch.Size([768, 768]) False
model.vlm_with_expert.vlm.model.vision_model.encoder.layers.0.self_attn.v_proj.bias torch.Size([768]) False
model.vlm_with_expert.vlm.model.vision_model.encoder.layers.0.self_attn.q_proj.weight torch.Size([768, 768]) False
model.vlm_with_expert.vlm.model.vision_model.encoder.layers.0.self_attn.q_proj.bias torch.Size([768]) False
mode

In [15]:
print("total params:", sum(p.numel() for p in policy.parameters()) / 1000000, "mil")
print("trainable params:", sum(p.numel() for p in policy.parameters() if p.requires_grad) / 1000000, "mil")

total params: 450.046176 mil
trainable params: 99.880992 mil


In [ ]:
import torch

lr = 1e-4
weight_decay = 0.01

opt = torch.optim.AdamW([p.numel() for p in policy.parameters() if p.requires_grad], lr = lr, weight_decay = weight_decay)


#for batch in train_loader:
    #here we can have the usual training loop 
    #we deffo need some checkpoints here though 

In [7]:
import gymnasium as gym
import mani_skill.envs

env = gym.make(
    "PushCube-v1", # there are more tasks e.g. "PickCube-v1", "PegInsertionSide-v1", ...
    num_envs=1,
    obs_mode="state", # there is also "state_dict", "rgbd", ...
    control_mode="pd_ee_delta_pose", # there is also "pd_joint_delta_pos", ...
    render_mode = "rgb_array"
)
print("Observation space", env.observation_space)
print("Action space", env.action_space)

obs, _ = env.reset(seed=0) # reset with a seed for determinism
done = False
while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    env.render()  # a display is required to render
env.close()

Observation space Box(-inf, inf, (1, 35), float32)
Action space Box(-1.0, 1.0, (7,), float32)


In [ ]:
import gymnasium as gym
import torch
from collections import defaultdict
from mani_skill.vector.wrappers.gymnasium import ManiSkillVectorEnv
env_id = "PushCube-v1"
num_eval_envs = 64
env_kwargs = dict(obs_mode="state") # modify your env_kwargs here
eval_envs = gym.make(env_id, num_envs=num_eval_envs, reconfiguration_freq=1, **env_kwargs)
# add any other wrappers here
eval_envs = ManiSkillVectorEnv(eval_envs, ignore_terminations=True, record_metrics=True)

# evaluation loop, which will record metrics for complete episodes only
obs, _ = eval_envs.reset(seed=0)
eval_metrics = defaultdict(list)
for _ in range(400):
    action = eval_envs.action_space.sample() # replace with your policy action
    obs, rew, terminated, truncated, info = eval_envs.step(action)
    # note as there are no partial resets, truncated is True for all environments at the same time
    if truncated.any():
        for k, v in info["final_info"]["episode"].items():
            eval_metrics[k].append(v.float())
for k in eval_metrics.keys():
    print(f"{k}_mean: {torch.mean(torch.stack(eval_metrics[k])).item()}")
    